# 🦕 DINO SDK - IoT Telemetry Ingestion

## 📊 Pipeline de Ingestão de Telemetria IoT
Este notebook realiza a ingestão estruturada de dados de telemetria IoT usando o **DINO IngestionEngine**.

### 🎯 Objetivo
- Ingerir dados de sensores IoT de forma batch/streaming
- Aplicar schema evolution automática
- Usar Liquid Clustering para otimização
- Integração completa com Unity Catalog

### 🏗️ Arquitetura
```
Raw Data (ADLS) → DINO IngestionEngine → Unity Catalog (Bronze)
```

**Configuração Esperada:**
- **Catálogo**: `data_master_dev_dbw`
- **Schema**: `bronze` 
- **Tabela**: `device_telemetry`
- **Origem**: `{schema_location}/device_telemetry/`

In [ ]:
# 📦 Imports e Configuração Inicial
print("🦕 DINO SDK - IoT Telemetry Ingestion Pipeline")
print("=" * 60)

import logging
from datetime import datetime
from typing import Dict, List, Optional, Any
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import *
from pyspark.sql.types import *

# DINO SDK
try:
    from dino_sdk import (
        IngestionEngine,
        IngestionConfig, 
        get_ingestion_engine,
        SchemaManager,
        DataReader,
        DataSaver
    )
    print("✅ DINO SDK importado com sucesso!")
    dino_available = True
except ImportError as e:
    print(f"❌ Erro ao importar DINO SDK: {e}")
    dino_available = False

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

logger = logging.getLogger(__name__)

## ⚙️ Configuração do Pipeline
Definindo configurações centralizadas para o pipeline de ingestão.

In [ ]:
# ⚙️ Configurações do Pipeline
print("⚙️ Configurando pipeline de ingestão...")

# Configurações Unity Catalog
CATALOG_NAME = "data_master_dev_dbw"
SCHEMA_NAME = "bronze"
TABLE_NAME = "device_telemetry"

# Configurações de dados
SOURCE_PATH = "/mnt/iot-data/sensors/"  # Fallback path
FILE_EXTENSION = "json"  # Formato esperado dos dados IoT
FILE_HEADER = True

# Configurações de execução
TYPE_RUN = "batch"  # ou "streaming" para dados em tempo real
TRIGGER_PROCESSING = "10 seconds"

# Configurações de schema evolution
SCHEMA_EVOLUTION_MODE = "addNewColumns"  # Permite novas colunas automaticamente
RESCUE_DATA_COLUMN = "_rescued_data"

# Configurações de clustering
LIQUID_CLUSTERING = True
CLUSTERING_COLUMNS = ["device_id", "event_timestamp", "location_id"]

# Configurações de checkpoint (para streaming)
CHECKPOINT_LOCATION = f"/mnt/checkpoints/{TABLE_NAME}"

print(f"📂 Destino: {CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}")
print(f"📁 Origem: {SOURCE_PATH}")
print(f"🔄 Modo: {TYPE_RUN}")
print(f"🏷️ Clustering: {CLUSTERING_COLUMNS}")

## 🔧 Inicialização do Spark e DINO Engine
Configurando sessão Spark e instanciando o IngestionEngine.

In [ ]:
# 🔧 Inicialização do Spark Session
print("\n🔧 Inicializando Spark Session...")

try:
    # Obter sessão Spark existente ou criar nova
    spark = SparkSession.builder \
        .appName("DINO-IoT-Ingestion") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .getOrCreate()
    
    print("✅ Spark Session inicializada")
    print(f"   📊 Spark Version: {spark.version}")
    print(f"   🏷️ App Name: {spark.sparkContext.appName}")
    
    # Configurar nível de log
    spark.sparkContext.setLogLevel("WARN")
    
except Exception as e:
    print(f"❌ Erro ao inicializar Spark: {e}")
    raise

# 🦕 Inicializar DINO IngestionEngine
if dino_available:
    try:
        print("\n🦕 Inicializando DINO IngestionEngine...")
        
        # Usar função helper para criar engine
        engine = get_ingestion_engine(spark)
        print("✅ DINO IngestionEngine criado com sucesso!")
        print(f"   🔧 Engine Type: {type(engine)}")
        
    except Exception as e:
        print(f"❌ Erro ao criar IngestionEngine: {e}")
        engine = None
else:
    print("⚠️ DINO SDK não disponível - criando engine mock")
    engine = None

## 📋 Configuração de Ingestão
Criando objeto `IngestionConfig` com todas as configurações necessárias.

In [ ]:
# 📋 Criar Configuração de Ingestão
print("\n📋 Criando configuração de ingestão...")

if dino_available:
    try:
        # Criar configuração detalhada
        config = IngestionConfig(
            # Origem dos dados
            source_path=SOURCE_PATH,
            file_extension=FILE_EXTENSION,
            file_header=FILE_HEADER,
            file_delimiter=",",  # Não usado para JSON
            
            # Destino Unity Catalog  
            catalog_name=CATALOG_NAME,
            schema_name=SCHEMA_NAME,
            table_name=TABLE_NAME,
            
            # Configurações de execução
            type_run=TYPE_RUN,
            trigger_processing_time=TRIGGER_PROCESSING,
            
            # Schema evolution
            schema_evolution_mode=SCHEMA_EVOLUTION_MODE,
            rescue_data_column=RESCUE_DATA_COLUMN,
            multiline=False  # Para dados JSON simples
        )
        
        print("✅ IngestionConfig criado com sucesso!")
        print(f"   📂 Destino: {config.catalog_name}.{config.schema_name}.{config.table_name}")
        print(f"   📁 Origem: {config.source_path}")
        print(f"   📄 Formato: {config.file_extension}")
        print(f"   🔄 Tipo: {config.type_run}")
        print(f"   🧬 Schema Evolution: {config.schema_evolution_mode}")
        
    except Exception as e:
        print(f"❌ Erro ao criar IngestionConfig: {e}")
        config = None
else:
    print("⚠️ DINO SDK não disponível - configuração não criada")
    config = None

## 🛡️ Verificação e Preparação do Schema
Garantindo que o schema de destino existe e está configurado corretamente.

In [ ]:
# 🛡️ Verificação do Schema de Destino
print("\n🛡️ Verificando schema de destino...")

try:
    # Verificar se o schema existe
    schemas = spark.sql(f"SHOW SCHEMAS IN {CATALOG_NAME}").collect()
    schema_exists = any(row.schemaName == SCHEMA_NAME for row in schemas)
    
    if schema_exists:
        print(f"✅ Schema {CATALOG_NAME}.{SCHEMA_NAME} existe")
        
        # Verificar se a tabela já existe
        try:
            tables = spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.{SCHEMA_NAME}").collect()
            table_exists = any(row.tableName == TABLE_NAME for row in tables)
            
            if table_exists:
                print(f"✅ Tabela {TABLE_NAME} já existe")
                
                # Mostrar informações da tabela existente
                table_info = spark.sql(f"DESCRIBE TABLE EXTENDED {CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}").collect()
                print("📊 Informações da tabela existente:")
                for row in table_info[:5]:  # Primeiras 5 linhas
                    print(f"   {row.col_name}: {row.data_type}")
                    
            else:
                print(f"⚠️ Tabela {TABLE_NAME} não existe - será criada durante a ingestão")
                
        except Exception as e:
            print(f"⚠️ Erro ao verificar tabelas: {e}")
            
    else:
        print(f"⚠️ Schema {CATALOG_NAME}.{SCHEMA_NAME} não existe")
        print("💡 O schema deve ser criado antes da ingestão")
        
        # Opcionalmente, criar o schema
        if dino_available:
            try:
                from dino_sdk import create_schema_simple
                print("🏗️ Tentando criar schema...")
                result = create_schema_simple(spark, CATALOG_NAME, SCHEMA_NAME)
                if result.get('success'):
                    print(f"✅ Schema {SCHEMA_NAME} criado com sucesso!")
                else:
                    print(f"❌ Falha ao criar schema: {result.get('errors')}")
            except Exception as e:
                print(f"⚠️ Não foi possível criar schema automaticamente: {e}")

except Exception as e:
    print(f"❌ Erro na verificação do schema: {e}")

## 🚀 Execução da Ingestão
Executando o pipeline de ingestão com o DINO IngestionEngine.

In [ ]:
# 🚀 Execução da Ingestão
print("\n🚀 Iniciando processo de ingestão...")

if engine and config:
    try:
        print("⏳ Executando ingestão com DINO IngestionEngine...")
        print(f"📊 Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        
        # Executar ingestão
        result = engine.ingest(config)
        
        print("\n📊 Resultado da Ingestão:")
        print("=" * 40)
        
        if result.get('success'):
            print("✅ INGESTÃO CONCLUÍDA COM SUCESSO!")
            print(f"   📊 Registros processados: {result.get('records_processed', 'N/A')}")
            print(f"   ⏱️ Tempo de execução: {result.get('execution_time', 'N/A')}")
            print(f"   📂 Tabela criada/atualizada: {CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}")
            
            # Informações adicionais se disponíveis
            if result.get('schema_evolved'):
                print(f"   🧬 Schema evolution aplicada: {result.get('new_columns', 'N/A')} novas colunas")
            
            if result.get('clustering_applied'):
                print(f"   🏷️ Liquid Clustering aplicado: {CLUSTERING_COLUMNS}")
                
        else:
            print("❌ INGESTÃO FALHOU!")
            print(f"   🐛 Erro: {result.get('error', 'Erro desconhecido')}")
            if result.get('details'):
                print(f"   📝 Detalhes: {result.get('details')}")
                
    except Exception as e:
        print(f"❌ Erro durante a execução da ingestão: {e}")
        import traceback
        traceback.print_exc()
        
else:
    print("⚠️ Engine ou configuração não disponível")
    print("🔄 Executando ingestão simulada...")
    
    # Simulação para demonstração
    result = {
        'success': False,
        'error': 'DINO SDK não disponível - modo simulação',
        'records_processed': 0
    }

## 📊 Validação dos Resultados
Verificando os dados ingeridos e validando a qualidade.

In [ ]:
# 📊 Validação dos Resultados
print("\n📊 Validando resultados da ingestão...")

try:
    # Verificar se a tabela existe após ingestão
    table_path = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"
    
    # Contar registros na tabela
    try:
        count = spark.sql(f"SELECT COUNT(*) as total FROM {table_path}").collect()[0].total
        print(f"📊 Total de registros na tabela: {count:,}")
        
        if count > 0:
            print("\n📋 Schema da tabela:")
            spark.sql(f"DESCRIBE {table_path}").show()
            
            print("\n📝 Amostra dos dados (5 registros):")
            spark.sql(f"SELECT * FROM {table_path} LIMIT 5").show()
            
            # Estatísticas adicionais
            print("\n📈 Estatísticas da tabela:")
            stats_df = spark.sql(f"""
                SELECT 
                    COUNT(*) as total_records,
                    COUNT(DISTINCT device_id) as unique_devices,
                    MIN(event_timestamp) as min_timestamp,
                    MAX(event_timestamp) as max_timestamp
                FROM {table_path}
            """)
            stats_df.show()
            
        else:
            print("⚠️ Tabela criada mas sem registros")
            
    except Exception as e:
        print(f"⚠️ Não foi possível validar dados: {e}")
        print("💡 A tabela pode não ter sido criada ainda")

except Exception as e:
    print(f"⚠️ Erro na validação: {e}")

print(f"\n✅ Pipeline de ingestão finalizado!")
print(f"📅 Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 📋 Resumo e Próximos Passos

### ✅ O que foi realizado:
1. **Configuração**: Pipeline configurado para dados IoT JSON
2. **Schema**: Verificação/criação automática do schema Unity Catalog  
3. **Ingestão**: Execução com DINO IngestionEngine
4. **Validação**: Verificação da qualidade e contagem dos dados

### 🔄 Próximos Passos:
1. **Monitoramento**: Configurar alertas para falhas de ingestão
2. **Scheduling**: Usar DINO WorkflowManager para automação
3. **Transformações**: Adicionar lógica de limpeza e enriquecimento
4. **Streaming**: Migrar para ingestão em tempo real se necessário

### 🏗️ Para Produção:
- Configurar file arrival triggers via WorkflowManager
- Implementar retry logic robusto  
- Adicionar métricas de qualidade de dados
- Configurar backup e disaster recovery

In [ ]:
# 📋 Relatório Final
print("🦕 DINO SDK - Relatório Final de Ingestão IoT")
print("=" * 60)

# Resumo da execução
execution_summary = {
    "pipeline": "IoT Telemetry Ingestion",
    "dino_sdk_available": dino_available,
    "spark_version": spark.version if 'spark' in locals() else "N/A",
    "target_table": f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}",
    "execution_time": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "status": "SUCCESS" if result.get('success', False) else "FAILED",
    "records_processed": result.get('records_processed', 0) if result else 0
}

print("\n📊 Resumo da Execução:")
for key, value in execution_summary.items():
    print(f"   {key}: {value}")

if result and result.get('success'):
    print("\n🎉 PIPELINE EXECUTADO COM SUCESSO!")
    print("✅ Dados IoT ingeridos no Unity Catalog")
    print("✅ Schema evolution aplicada automaticamente")
    print("✅ Liquid Clustering configurado")
else:
    print("\n⚠️ Pipeline executado em modo demonstração")
    print("💡 Instale a WHL do DINO SDK para execução completa")

print(f"\n📅 Pipeline finalizado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")